# 01 — Dataset preparation

Loads the DRAMA annotations, checks their quality, matches them against the frames
available on disk, and builds the balanced evaluation subset. Also verifies how much of
the annotated vocabulary the action taxonomy covers, and consolidates the run outputs
into a single results file once the pipeline has been executed.

Requires `skg.py`. Writes `datos.py`, used by notebooks 02 and 03.

## The data module

Two design decisions worth flagging. Scenarios are matched on the full relative path rather
than the filename, because names like `frame_000786.png` repeat across clips and matching on
the name alone would pair the wrong annotation with the wrong image. And the scenario list is
sorted before sampling, since `os.walk` gives no ordering guarantee and the same seed would
otherwise pick different scenarios on different runs.

In [ ]:
%%writefile ../src/datos.py
"""DRAMA loading and stratified sampling.

Matches the annotations against the frames actually extracted on disk and
builds the evaluation batches. No GPU needed.
"""

import os
import json
import random

# ---------------------------------------------------------------------------
# Paths: set the environment variables or place the data under ../data/drama
# ---------------------------------------------------------------------------
RUTA_ANOTACIONES = os.environ.get(
    "DRAMA_ANNOTATIONS", "../data/drama/integrated_output_v2.json")
BASE_IMAGENES = os.environ.get("DRAMA_FRAMES", "../data/drama/combined")

TIPOS = ["Vehicle", "Pedestrian", "Cyclist", "Infrastructure"]


def cargar_escenarios(ruta_anotaciones=RUTA_ANOTACIONES, base_imagenes=BASE_IMAGENES):
    """Return the annotated scenarios whose frame is available on disk.

    Matching is done on the relative path (folder + filename), not the filename
    alone: names like 'frame_000786.png' repeat across clips.
    """
    with open(ruta_anotaciones) as f:
        data = json.load(f)

    # 1) Inventory of frames on disk, as a set of relative paths.
    #    A set gives O(1) lookups, which matters with thousands of images.
    rutas_en_disco = set()
    for carpeta, _, archivos in os.walk(base_imagenes):
        for archivo in archivos:
            if archivo.startswith("frame_") and archivo.endswith(".png"):
                completa = os.path.join(carpeta, archivo)
                if "/combined/" in completa:
                    rutas_en_disco.add(completa.split("/combined/")[1])

    # 2) Apply the same transformation to each annotation's S3 URL.
    #    That symmetry is what makes the match correct.
    escenarios = []
    for i, esc in enumerate(data):
        url = esc.get("s3_fileUrl", "")
        if "/combined/" not in url:
            continue
        rel = url.split("/combined/")[1]
        if rel not in rutas_en_disco:
            continue
        # Skip the no-risk partition, which carries N/A in these fields
        if any(str(esc.get(c, "")).strip().upper() == "N/A"
               for c in ["Agent-classifier", "Caption", "Suggestions (to the ego car)"]):
            continue
        escenarios.append({
            "indice": i,
            "tipo": esc.get("Agent-classifier"),
            "ruta": os.path.join(base_imagenes, rel),
            "caption": esc.get("Caption", ""),
            "sugerencia": esc.get("Suggestions (to the ego car)", ""),
        })

    # 3) Deterministic order: os.walk gives no guarantee, and without this the
    #    same seed would pick different scenarios across sessions.
    escenarios.sort(key=lambda e: e["indice"])
    return escenarios


def muestreo_estratificado(escenarios, n_por_tipo, seed, excluir=None):
    """Take n scenarios from each agent class, with a fixed seed.

    `excluir` holds indices already used, for extension batches.
    """
    excluir = excluir or set()
    random.seed(seed)
    lote = []
    for tipo in TIPOS:
        candidatos = [e for e in escenarios
                      if e["tipo"] == tipo and e["indice"] not in excluir]
        lote.extend(random.sample(candidatos, min(n_por_tipo, len(candidatos))))
    return lote


def intercalar_por_tipo(lote):
    """Reorder the batch alternating classes (V, P, C, I, V, P, ...).

    Useful when a run may be cut short: wherever it stops, the sample stays
    balanced across classes.
    """
    por_tipo = {t: [e for e in lote if e["tipo"] == t] for t in TIPOS}
    maximo = max((len(v) for v in por_tipo.values()), default=0)
    salida = []
    for i in range(maximo):
        for t in TIPOS:
            if i < len(por_tipo[t]):
                salida.append(por_tipo[t][i])
    return salida


def frames_del_escenario(ruta_frame_anotado):
    """All frames of the clip the annotated frame belongs to, in order."""
    carpeta = os.path.dirname(ruta_frame_anotado)
    frames = [os.path.join(carpeta, f) for f in os.listdir(carpeta)
              if f.startswith("frame_") and f.endswith(".png")]
    # Alphabetical order is temporal order, since names are zero-padded
    return sorted(frames)


def muestrear_frames(frames, n, frame_anotado=None):
    """Pick n evenly spaced frames, making sure the annotated one is included."""
    if n >= len(frames):
        seleccion = list(frames)
    elif n == 1:
        seleccion = [frame_anotado or frames[-1]]
    else:
        # n positions spread between the first and the last
        idxs = [round(i * (len(frames) - 1) / (n - 1)) for i in range(n)]
        seleccion = [frames[i] for i in idxs]

    # The annotated frame is the one the ground truth refers to: it must be there
    if frame_anotado and frame_anotado not in seleccion:
        seleccion[-1] = frame_anotado
    return seleccion

In [ ]:
import sys, os, json, re, random, time
from collections import Counter, defaultdict

sys.path.insert(0, "../src")   # project modules
from skg import SKG, CLASE_DE, normalizar_accion
from datos import (cargar_escenarios, muestreo_estratificado, intercalar_por_tipo,
                   RUTA_ANOTACIONES, TIPOS)

RESULTADOS = os.environ.get("RESULTS_DIR", "../results")

data = json.load(open(RUTA_ANOTACIONES))
print(f"Annotated scenarios in DRAMA: {len(data)}")

## What the annotations contain

DRAMA annotates at three levels: questions about the video, the bounding box of the risk
object, and questions about that object. The pipeline uses four of those fields.

In [ ]:
for k, v in list(data[0].items())[:12]:
    print(f"  {k:<32} {str(v)[:60]}")

print()
print("Risk:", dict(Counter(d.get("Risk") for d in data)))
print("Agent class:", dict(Counter(d.get("Agent-classifier") for d in data)))
print("Ego intention:", dict(Counter(d.get("Intention of Ego-vehicle") for d in data
                                     if d.get("Risk") == "Yes")))

## Data quality

DRAMA writes `"N/A"` instead of leaving fields empty, so the usual null checks miss them.
The records without an agent class turn out to be exactly the ones without a confirmed risk,
which is the negative partition of the dataset rather than corrupted data.

In [ ]:
campos = ["Caption", "Suggestions (to the ego car)", "Agent-classifier",
          "s3_fileUrl", "geometry", "Risk"]

print(f"{'field':<32}{'None':>6}{'empty':>7}{'N/A':>7}")
for campo in campos:
    nones  = sum(1 for d in data if d.get(campo) is None)
    vacios = sum(1 for d in data if isinstance(d.get(campo), (str, list)) and len(d.get(campo)) == 0)
    nas    = sum(1 for d in data if isinstance(d.get(campo), str)
                 and d.get(campo).strip().upper() == "N/A")
    print(f"{campo:<32}{nones:>6}{vacios:>7}{nas:>7}")

print("\nRisk vs missing agent class:",
      dict(Counter((d.get("Risk"), d.get("Agent-classifier") == "N/A") for d in data)))

urls = [d.get("s3_fileUrl") for d in data]
print(f"Duplicate image URLs: {len(urls) - len(set(urls))}")

# Boxes that are missing, malformed, or have no area
malas = 0
for d in data:
    g = d.get("geometry")
    if not g or not isinstance(g, list) or len(g) < 3:
        malas += 1
        continue
    xs, ys = [p[0] for p in g], [p[1] for p in g]
    if max(xs) - min(xs) <= 0 or max(ys) - min(ys) <= 0:
        malas += 1
print(f"Missing or degenerate bounding boxes: {malas}/{len(data)}")

## Matching against the extracted frames

Only part of DRAMA was extracted locally, so the pool of usable scenarios is whatever
survives the match. The filtering happens here rather than afterwards: scenarios without a
confirmed risk never enter the pool.

In [ ]:
escenarios = cargar_escenarios()
print(f"Scenarios with an available frame: {len(escenarios)}")
print(dict(Counter(e["tipo"] for e in escenarios)))

# The paired tests in Chapter 3 assume independent scenarios, so no two of them
# should come from the same clip.
clips = Counter(os.path.basename(os.path.dirname(e["ruta"])) for e in escenarios)
print(f"Unique clips: {len(clips)} | clips contributing more than one: "
      f"{sum(1 for v in clips.values() if v > 1)}")

## Image resolution

The subset mixes two camera resolutions, which is what motivates capping the visual token
budget in notebook 02. Without the cap a large frame would produce several times more tokens
than a small one and the conditions would not be comparable.

In [ ]:
from PIL import Image

resoluciones = Counter()
for e in escenarios:
    with Image.open(e["ruta"]) as im:
        resoluciones[im.size] += 1
for tam, n in resoluciones.most_common():
    print(f"  {tam[0]}x{tam[1]}  {n:>4} scenarios  ({tam[0]*tam[1]/1e6:.1f} MP)")

## Selection bias

The subset is balanced by class while DRAMA is not, which is what the reweighted estimate in
Section 3.1 corrects for.

In [ ]:
total    = Counter(d.get("Agent-classifier") for d in data if d.get("Risk") == "Yes")
extraido = Counter(e["tipo"] for e in escenarios)
n_total, n_ext = sum(total[t] for t in TIPOS), sum(extraido[t] for t in TIPOS)

print(f"{'class':<18}{'DRAMA':>8}{'%':>8}{'subset':>9}{'%':>8}")
for t in TIPOS:
    print(f"{t:<18}{total[t]:>8}{100*total[t]/n_total:>7.1f}%"
          f"{extraido[t]:>9}{100*extraido[t]/n_ext:>7.1f}%")

## Taxonomy coverage

The six canonical labels were derived from the vocabulary of the annotated suggestions rather
than defined beforehand. What falls outside are permissive instructions, which the graph does
not represent by design: the Vienna Convention codifies restrictive obligations and prescribes
no duty to resume motion.

In [ ]:
sugerencias = [d["Suggestions (to the ego car)"] for d in data
               if d.get("Risk") == "Yes"
               and d.get("Suggestions (to the ego car)", "N/A") != "N/A"]

cobertura = Counter(normalizar_accion(s)[0] for s in sugerencias)
sin_mapear = cobertura.pop(None, 0)
n = len(sugerencias)

print(f"Risk-positive suggestions: {n}\n")
for accion, cuenta in cobertura.most_common():
    print(f"  {100*cuenta/n:>5.1f}%  {cuenta:>6}  {accion}")
print(f"  {100*sin_mapear/n:>5.1f}%  {sin_mapear:>6}  [not covered]")

fuera = Counter(s.strip().lower() for s in sugerencias if normalizar_accion(s)[0] is None)
print("\nUncovered:", fuera.most_common(4))

### Figure — taxonomy coverage

In [ ]:
import matplotlib.pyplot as plt

pares = cobertura.most_common()
etiquetas = [k for k, _ in pares][::-1] + ["Not covered"]
valores   = [100*v/n for _, v in pares][::-1] + [100*sin_mapear/n]
colores   = ["#2F4B7C"]*len(pares) + ["#A8B8D0"]

fig, ax = plt.subplots(figsize=(7, 3.6))
barras = ax.barh(etiquetas, valores, color=colores, height=0.65)
for bar, v in zip(barras, valores):
    ax.text(v + 0.6, bar.get_y() + bar.get_height()/2, f"{v:.1f}%", va="center", fontsize=9)
ax.set_xlabel("Share of annotated suggestions (%)")
ax.set_xlim(0, max(valores)*1.18)
ax.grid(axis="x", alpha=0.25); ax.set_axisbelow(True)
for s in ["top", "right"]: ax.spines[s].set_visible(False)
plt.tight_layout()
fig.savefig("fig_cobertura_taxonomia.pdf", bbox_inches="tight", dpi=300)
plt.show()

## Action distribution per class

Part of why the graph performs better on some classes than others: where the annotation
concentrates on one action, a deterministic rule hits more often.

In [ ]:
tabla = defaultdict(Counter)
for d in data:
    if d.get("Risk") != "Yes":
        continue
    accion = normalizar_accion(d.get("Suggestions (to the ego car)", ""))[0]
    if d.get("Agent-classifier") in TIPOS and accion:
        tabla[d["Agent-classifier"]][accion] += 1

for clase in TIPOS:
    cuenta = tabla[clase]
    total_clase = sum(cuenta.values())
    print(f"\n{clase} (n={total_clase})")
    for accion, v in cuenta.most_common():
        print(f"   {100*v/total_clase:>5.1f}%  {accion}")

## The evaluation subset

130 scenarios per class, with the seed fixed so the selection is reproducible. The batch is
interleaved by class so that if a run is cut short by the time limit, whatever was processed
stays balanced.

In [ ]:
lote = intercalar_por_tipo(muestreo_estratificado(escenarios, 130, seed=123))
print(f"Evaluation subset: {len(lote)}", dict(Counter(e["tipo"] for e in lote)))
print("First few, interleaved:", [e["tipo"][:4] for e in lote[:8]])

> The subset was actually built in three batches, as more clips were extracted from the
> DRAMA archive. The cell above reproduces the sampling over the full pool. The indices that
> were evaluated are recorded in the result files.

---

## Consolidating the results

Run this **after** notebook 02. It merges the per-condition files into one record per
scenario, which is what notebook 03 reads.

In [ ]:
def consolidar():
    """Merge the per-condition result files into one record per scenario."""

    def cargar(nombre):
        try:
            return json.load(open(f"{RESULTADOS}/{nombre}"))
        except FileNotFoundError:
            print(f"  [missing] {nombre}")
            return []

    principal = cargar("pool520_1f.json")
    idx_B = {r["indice"]: r for r in cargar("pool520_sin_conocimiento.json")}
    idx_T = {r["indice"]: r for r in cargar("pool520_10f.json")}
    idx_O = {r["indice"]: r for r in cargar("pool520_bbox.json")}

    caps_A = {r["indice"]: r["caption"] for r in cargar("pool520_captions_fewshot.json")}
    caps_B = {r["indice"]: r["caption"] for r in cargar("pool520_B_captions.json")}
    validacion = {r["indice"]: r["validacion"]
                  for r in cargar("pool520_validacion.json") if r.get("validacion")}
    juez = {r["indice"]: r["juicio"]
            for r in cargar("juicio_qwen_520.json") if r.get("juicio")}

    final = []
    for r in principal:
        i = r["indice"]
        final.append({
            "indice": i,
            "tipo": r["tipo"],
            "caption_gt": r["caption_gt"],
            "sugerencia_gt": r["sugerencia_gt"],
            "A_con_skg": {**r["con_skg"], "percepcion": r.get("percepcion"),
                          "caption_fewshot": caps_A.get(i, "")},
            "B_sin_conocimiento": idx_B[i]["con_skg"] if i in idx_B else None,
            "B_caption": caps_B.get(i, ""),
            "C_baseline": r.get("sin_skg"),
            # The perception of each variant is kept, otherwise the funnel analysis
            # in notebook 03 silently falls back on the single-frame one.
            "temporal_10f": ({**idx_T[i]["con_skg"], "percepcion": idx_T[i].get("percepcion")}
                             if i in idx_T else None),
            "oraculo_bbox": ({**idx_O[i]["con_skg"], "percepcion": idx_O[i].get("percepcion")}
                             if i in idx_O else None),
            "validacion": validacion.get(i),
            "juicio_automatico": juez.get(i),
        })
    return final


final = consolidar()
salida = f"{RESULTADOS}/resultados_finales.json"
json.dump(final, open(salida, "w"), indent=2, ensure_ascii=False)

print(f"Saved: {salida}  ({os.path.getsize(salida)/1e6:.1f} MB)")
print(f"Scenarios: {len(final)}", dict(Counter(r["tipo"] for r in final)))
for k in ["B_sin_conocimiento", "temporal_10f", "oraculo_bbox",
          "validacion", "juicio_automatico"]:
    print(f"  {k:<22} {sum(1 for r in final if r.get(k))}/{len(final)}")

Two gaps are expected. Validation misses one scenario, where the router emitted a hazard
outside the catalogue and no explanation was ever generated. The image verification misses that
same one plus another whose output could not be parsed.